In [14]:
%cd ..
import os
import torch

from base64 import b64encode
from cotracker.utils.visualizer import Visualizer, read_video_from_path
from IPython.display import HTML

/home/mhzou


/home/mhzou/conda/envs/pytorch/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [15]:
import torch.nn.functional as F

video_np = read_video_from_path("/home/mhzou/co-tracker/assets/front_camera_view.mp4")
if video_np is None:
    raise FileNotFoundError("Could not read video file. Check the path and file permissions.")

video = torch.from_numpy(video_np).permute(0, 3, 1, 2)[None].float()

# Memory controls: cap frame count and resize long side
max_frames = 120
if video.shape[1] > max_frames:
    frame_idx = torch.linspace(0, video.shape[1] - 1, max_frames).long()
    video = video[:, frame_idx]

b, t, c, h, w = video.shape
max_side = 512
if max(h, w) > max_side:
    scale = max_side / max(h, w)
    new_h, new_w = int(h * scale), int(w * scale)
    video_4d = video.view(b * t, c, h, w)
    video_4d = F.interpolate(video_4d, size=(new_h, new_w), mode="bilinear", align_corners=False)
    video = video_4d.view(b, t, c, new_h, new_w)

In [16]:
def show_video(video_path):
    video_file = open(video_path, "r+b").read()
    video_url = f"data:video/mp4;base64,{b64encode(video_file).decode()}"
    return HTML(f"""<video width="640" height="480" autoplay loop controls><source src="{video_url}"></video>""")
 
show_video("/home/mhzou/co-tracker/assets/front_camera_view.mp4")

In [ ]:
from cotracker.predictor import CoTrackerPredictor

model = CoTrackerPredictor(
    checkpoint=os.path.join(
        '/home/mhzou/co-tracker/checkpoints/scaled_offline.pth'
    )
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
video_in = video
try:
    model = model.to(device)
    video_in = video.to(device, non_blocking=True)

    with torch.inference_mode():
        if device == 'cuda':
            torch.cuda.empty_cache()
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                pred_tracks, pred_visibility = model(video_in, grid_size=1)
        else:
            pred_tracks, pred_visibility = model(video_in, grid_size=1)

except torch.cuda.OutOfMemoryError:
    print('CUDA OOM detected. Retrying on CPU with stronger downsampling...')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = model.cpu()
    video_in = video[:, ::2, :, ::2, ::2].cpu()

    with torch.inference_mode():
        pred_tracks, pred_visibility = model(video_in, grid_size=1)

vis = Visualizer(save_dir='./videos', pad_value=100)
vis.visualize(video=video_in, tracks=pred_tracks, visibility=pred_visibility, filename='teaser')

show_video('./videos/teaser.mp4')

CUDA OOM detected. Retrying on CPU with stronger downsampling...
